# Paired bootstrap: mean difference fusion vs image-only

Tests **test macro-F1** and **test accuracy** on 30 matched seeds.

In [1]:
import json
import os
import re
import statistics
from pathlib import Path

import numpy as np

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)


FUSION_EXP = "phase3_robustness"
IMAGE_EXP = "imageonly_robustness"
ALPHA = 0.05
B = 10_000
RNG_SEED = 42

METRICS = [
    ("test_macro_f1", "Test Macro F1", False),
    ("test_accuracy", "Test Accuracy", True),
]


def load_test_metric(exp_name: str, json_key: str, as_fraction: bool = False) -> dict[int, float]:
    out: dict[int, float] = {}
    base = PROJECT_ROOT / "experiments" / exp_name / "metrics" / "experiments"
    for path in base.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        obj = json.loads(path.read_text(encoding="utf-8"))
        v = float(obj["test_metrics"][json_key])
        if as_fraction and v > 1.0:
            v /= 100.0
        out[int(m.group(1))] = v
    return out


for json_key, label, as_fraction in METRICS:
    fusion = load_test_metric(FUSION_EXP, json_key, as_fraction)
    image = load_test_metric(IMAGE_EXP, json_key, as_fraction)
    seeds = sorted(fusion.keys() & image.keys())
    if fusion.keys() != image.keys():
        raise ValueError(f"{label} seed mismatch")

    d = np.asarray([fusion[s] - image[s] for s in seeds], dtype=np.float64)
    n = d.shape[0]
    mean_d = float(np.mean(d))
    std_d = float(np.std(d, ddof=1)) if n > 1 else float("nan")
    cohen_d = mean_d / std_d if std_d else float("nan")
    wins = int(np.sum(d > 0))
    ties = int(np.sum(d == 0))
    losses = int(np.sum(d < 0))

    rng = np.random.default_rng(RNG_SEED)
    idx = rng.integers(0, n, size=(B, n))
    boot_means = d[idx].mean(axis=1)

    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    one_sided_lb = float(np.percentile(boot_means, 5.0))
    p_one_sided = (1 + int(np.sum(boot_means <= 0.0))) / (B + 1)

    print("=" * 60)
    print(label)
    print("fusion_mean", float(np.mean(list(fusion.values()))))
    print("image_mean", float(np.mean(list(image.values()))))
    print("n", n)
    print("mean_diff", mean_d)
    print("stdev_diff", std_d)
    print("paired_Cohen_d", cohen_d)
    print("wins_ties_losses", wins, ties, losses)
    print("median_diff", float(np.median(d)))
    print("B", B, "rng_seed", RNG_SEED)
    print("ci95_mean_diff", float(ci_low), float(ci_high))
    print("one_sided_95_lower_bound_mean_diff", one_sided_lb)
    print("bootstrap_p_one_sided_mean_le_0", p_one_sided)
    print("reject_H0_mean_le_0", p_one_sided < ALPHA)
    print("one_sided_lb_gt_0", one_sided_lb > 0.0)
    print()

Test Macro F1
fusion_mean 0.8310843219571424
image_mean 0.8145000550248404
n 30
mean_diff 0.016584266932301864
stdev_diff 0.010423274665449107
paired_Cohen_d 1.5910802952622087
wins_ties_losses 27 0 3
median_diff 0.017163615028558676
B 10000 rng_seed 42
ci95_mean_diff 0.012822994356700241 0.02022590475079399
one_sided_95_lower_bound_mean_diff 0.013488707598288314
bootstrap_p_one_sided_mean_le_0 9.999000099990002e-05
reject_H0_mean_le_0 True
one_sided_lb_gt_0 True

Test Accuracy
fusion_mean 0.831183879093199
image_mean 0.8152549094359615
n 30
mean_diff 0.015928969657237675
stdev_diff 0.010019102868218464
paired_Cohen_d 1.5898598773514805
wins_ties_losses 27 0 3
median_diff 0.017104928376968698
B 10000 rng_seed 42
ci95_mean_diff 0.012284473012737378 0.01939788129603545
one_sided_95_lower_bound_mean_diff 0.012932574507353401
bootstrap_p_one_sided_mean_le_0 9.999000099990002e-05
reject_H0_mean_le_0 True
one_sided_lb_gt_0 True

